# Global feedback-distribution comparison

Compare empirical commentary with a separately labelled synthetic supplement, then contrast the original and supplemented pools with the complete policy corpus. This is a pooled analysis across available sources, not a national comparison.

The original lexical, embedding and classifier distance formulas remain unchanged.


In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

from scipy.spatial.distance import jensenshannon
from scipy.linalg import sqrtm

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
# Restore the stable internal column interface at the input boundary.
SOURCE_SCHEMA = {'record_ref': 'doc_id', 'source_label': 'filename', 'source_path': 'source_file', 'relative_path': 'relative_source_file', 'collection': 'corpus', 'origin_type': 'source_type', 'augmentation_type': 'synthetic_type', 'geography': 'country', 'document_characters': 'char_count', 'document_words': 'word_count', 'segment_key': 'chunk_id', 'segment_index': 'chunk_index', 'section_context': 'heading_context', 'segment_content': 'chunk_text', 'segment_characters': 'chunk_char_count', 'segment_words': 'chunk_word_count'}


In [2]:
def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "data" / "policy_feedback_segments.csv").exists():
            return candidate
    raise FileNotFoundError("Could not locate data/chunks_all.csv")

PROJECT_ROOT = find_project_root()

INPUT_CSV = PROJECT_ROOT / "data" / "policy_feedback_segments.csv"
OUT_DIR = PROJECT_ROOT / "causal_semantics" / "output" / "distance"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Input CSV:", INPUT_CSV)
print("Output folder:", OUT_DIR)


Project root: /home/mouelhis/Private/github/new_dsac/cnlp
Input CSV: /home/mouelhis/Private/github/new_dsac/cnlp/data/policy_feedback_segments.csv
Output folder: /home/mouelhis/Private/github/new_dsac/cnlp/causal_semantics/output/distance


## Load and organize the text segments

Read the combined corpus and restore the pipeline’s established internal feature names. Identify the pooled Global policy material, empirical commentary, and synthetic supplement without rewriting the underlying text.


In [3]:
df = pd.read_csv(INPUT_CSV).rename(columns=SOURCE_SCHEMA)

df["chunk_text"] = df["chunk_text"].fillna("").astype(str)
df["country_norm"] = df["country"].fillna("").astype(str).str.lower()
df["source_file_norm"] = df["source_file"].fillna("").astype(str).str.lower()


def light_clean(text):
    text = str(text)

    # Decode common HTML artefacts
    text = (
        text.replace("&gt;", " ")
        .replace("&lt;", " ")
        .replace("&amp;", " and ")
    )

    # Remove extraction artefacts only
    text = re.sub(r"<!--.*?-->", " ", text, flags=re.DOTALL)
    text = re.sub(r"!\[[^\]]*\]\([^)]+\)", " ", text)
    text = re.sub(r"\S+\.(png|jpg|jpeg|webp|gif|svg)", " ", text, flags=re.IGNORECASE)

    # Remove Markdown table separator lines
    text = re.sub(
        r"^\s*\|?\s*:?-{3,}:?\s*(\|\s*:?-{3,}:?\s*)+\|?\s*$",
        " ",
        text,
        flags=re.MULTILINE,
    )

    # Remove very long separators
    text = re.sub(r"^\s*[-_=]{8,}\s*$", " ", text, flags=re.MULTILINE)

    # Normalize whitespace
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)

    return text.strip()

policy_pool = df[(df["corpus"] == "policy") & (df["source_type"] == "original")].copy()

orig_sentiment = df[
    (df["corpus"] == "sentiment")
    & (df["source_type"] == "original")
].copy()

syn_sentiment = df[
    (df["corpus"] == "sentiment")
    & (df["source_type"] == "synthetic")
].copy()

for subset in [policy_pool, orig_sentiment, syn_sentiment]:
    subset["text_for_eval"] = subset["chunk_text"].map(light_clean)

policy_pool = policy_pool[policy_pool["text_for_eval"].str.len() > 0].copy()
orig_sentiment = orig_sentiment[orig_sentiment["text_for_eval"].str.len() > 0].copy()
syn_sentiment = syn_sentiment[syn_sentiment["text_for_eval"].str.len() > 0].copy()

print("Global policy chunks:", len(policy_pool))
print("Global policy documents:", policy_pool["doc_id"].nunique())
print("Global policy characters:", policy_pool["text_for_eval"].str.len().sum())

print("\nOriginal sentiment chunks:", len(orig_sentiment))
print("Original sentiment documents:", orig_sentiment["doc_id"].nunique())
print("Original sentiment characters:", orig_sentiment["text_for_eval"].str.len().sum())

print("\nSynthetic sentiment chunks:", len(syn_sentiment))
print("Synthetic sentiment documents:", syn_sentiment["doc_id"].nunique())
print("Synthetic sentiment characters available:", syn_sentiment["text_for_eval"].str.len().sum())


Global policy chunks: 2052
Global policy documents: 56
Global policy characters: 3370085

Original sentiment chunks: 536
Original sentiment documents: 17
Original sentiment characters: 900307

Synthetic sentiment chunks: 212
Synthetic sentiment documents: 53
Synthetic sentiment characters available: 446379


## Determine the synthetic supplementation budget

Compare character totals before adding auxiliary records. Synthetic material is selected using the original deterministic sampling and cumulative-character rules.


In [4]:
policy_pool_chars = policy_pool["text_for_eval"].str.len().sum()
orig_sentiment_chars = orig_sentiment["text_for_eval"].str.len().sum()
syn_available_chars = syn_sentiment["text_for_eval"].str.len().sum()

required_synthetic_chars = max(policy_pool_chars - orig_sentiment_chars, 0)

syn_balanced = syn_sentiment.sample(frac=1, random_state=42).copy()
syn_balanced["cum_chars"] = syn_balanced["text_for_eval"].str.len().cumsum()
syn_balanced = syn_balanced[syn_balanced["cum_chars"] <= required_synthetic_chars].copy()

balanced_sentiment = pd.concat([orig_sentiment, syn_balanced], ignore_index=True)

print("Global policy characters:", policy_pool_chars)
print("Original sentiment characters:", orig_sentiment_chars)
print("Synthetic characters required:", required_synthetic_chars)
print("Synthetic characters available:", syn_available_chars)
print("Synthetic characters selected:", syn_balanced["text_for_eval"].str.len().sum())
print("Balanced sentiment characters:", balanced_sentiment["text_for_eval"].str.len().sum())
print("Difference Global policy - balanced sentiment:", policy_pool_chars - balanced_sentiment["text_for_eval"].str.len().sum())


Global policy characters: 3370085
Original sentiment characters: 900307
Synthetic characters required: 2469778
Synthetic characters available: 446379
Synthetic characters selected: 446379
Balanced sentiment characters: 1346686
Difference Global policy - balanced sentiment: 2023399


## Build matched comparison sets

Construct original and supplemented text samples while respecting their provenance and keeping the sampling seed fixed.


In [5]:
N = min(len(orig_sentiment), len(syn_balanced))

orig_eval = orig_sentiment.sample(n=N, random_state=42).copy()
syn_eval = syn_balanced.sample(n=N, random_state=42).copy()

texts_orig = orig_eval["text_for_eval"].tolist()
texts_syn = syn_eval["text_for_eval"].tolist()

texts_policy_global = policy_pool["text_for_eval"].tolist()
texts_sentiment_original = orig_sentiment["text_for_eval"].tolist()
texts_sentiment_balanced = balanced_sentiment["text_for_eval"].tolist()

print("Balanced original sentiment chunks:", len(texts_orig))
print("Balanced synthetic sentiment chunks:", len(texts_syn))
print("Global policy chunks:", len(texts_policy_global))
print("Balanced sentiment chunks:", len(texts_sentiment_balanced))


Balanced original sentiment chunks: 212
Balanced synthetic sentiment chunks: 212
Global policy chunks: 2052
Balanced sentiment chunks: 748


## Lexical comparison

Measure bag-of-words divergence and TF–IDF centroid similarity. These statistics detect distributional separation rather than document quality.


In [6]:
def js_ngram_distance(texts_a, texts_b, ngram_range=(1, 1), max_features=30000):
    vectorizer = CountVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=ngram_range,
        max_features=max_features,
        min_df=1,
    )

    X = vectorizer.fit_transform(texts_a + texts_b)

    A = np.asarray(X[:len(texts_a)].sum(axis=0)).ravel().astype(float)
    B = np.asarray(X[len(texts_a):].sum(axis=0)).ravel().astype(float)

    A = A / max(A.sum(), 1)
    B = B / max(B.sum(), 1)

    return float(jensenshannon(A, B, base=2) ** 2)


def tfidf_centroid_cosine(texts_a, texts_b):
    vectorizer = TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2),
        max_features=30000,
        min_df=1,
    )

    X = vectorizer.fit_transform(texts_a + texts_b)

    A = np.asarray(X[:len(texts_a)].mean(axis=0)).reshape(1, -1)
    B = np.asarray(X[len(texts_a):].mean(axis=0)).reshape(1, -1)

    return float(cosine_similarity(A, B)[0, 0])


lexical_metrics = pd.DataFrame([{
    "comparison": "original_sentiment_vs_selected_synthetic_sentiment_chunks",
    "unigram_js_divergence": js_ngram_distance(texts_orig, texts_syn, (1, 1)),
    "bigram_js_divergence": js_ngram_distance(texts_orig, texts_syn, (2, 2)),
    "tfidf_centroid_cosine": tfidf_centroid_cosine(texts_orig, texts_syn),
}])

lexical_metrics.to_csv(OUT_DIR / "global_chunk_level_lexical_distance.csv", index=False)
lexical_metrics


,comparison,unigram_js_divergence,bigram_js_divergence,tfidf_centroid_cosine
0,original_sentiment_vs_selected_synthetic_senti...,0.215477,0.562959,0.575896


## Embedding-based comparison

When the configured sentence embedding model is available, quantify the distance between representation distributions; otherwise this step records a missing metric.


In [7]:
try:
    from sentence_transformers import SentenceTransformer

    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

    emb_orig = model.encode(texts_orig, show_progress_bar=True, normalize_embeddings=True)
    emb_syn = model.encode(texts_syn, show_progress_bar=True, normalize_embeddings=True)

except Exception as e:
    emb_orig, emb_syn = None, None
    print("Sentence-transformers unavailable or failed:", e)


def frechet_distance(A, B, eps=1e-6):
    A = np.asarray(A, dtype=np.float64)
    B = np.asarray(B, dtype=np.float64)

    mu1, mu2 = A.mean(axis=0), B.mean(axis=0)

    cov1 = np.cov(A, rowvar=False) + np.eye(A.shape[1]) * eps
    cov2 = np.cov(B, rowvar=False) + np.eye(B.shape[1]) * eps

    covmean = sqrtm(cov1 @ cov2)

    if np.iscomplexobj(covmean):
        covmean = covmean.real

    return float(np.sum((mu1 - mu2) ** 2) + np.trace(cov1 + cov2 - 2 * covmean))


def rbf_mmd(A, B, gamma=None):
    A = np.asarray(A, dtype=np.float64)
    B = np.asarray(B, dtype=np.float64)

    if gamma is None:
        gamma = 1.0 / A.shape[1]

    Kxx = np.exp(-gamma * ((A[:, None, :] - A[None, :, :]) ** 2).sum(axis=2))
    Kyy = np.exp(-gamma * ((B[:, None, :] - B[None, :, :]) ** 2).sum(axis=2))
    Kxy = np.exp(-gamma * ((A[:, None, :] - B[None, :, :]) ** 2).sum(axis=2))

    return float(Kxx.mean() + Kyy.mean() - 2 * Kxy.mean())


if emb_orig is not None:
    sim = cosine_similarity(emb_orig, emb_syn)

    semantic_metrics = pd.DataFrame([{
        "comparison": "original_sentiment_vs_selected_synthetic_sentiment_chunks",
        "embedding_centroid_cosine": float(
            cosine_similarity(
                emb_orig.mean(axis=0, keepdims=True),
                emb_syn.mean(axis=0, keepdims=True),
            )[0, 0]
        ),
        "avg_nearest_synthetic_to_original_cosine": float(sim.max(axis=0).mean()),
        "avg_nearest_original_to_synthetic_cosine": float(sim.max(axis=1).mean()),
        "frechet_embedding_distance": frechet_distance(emb_orig, emb_syn),
        "mmd_rbf_embedding_distance": rbf_mmd(emb_orig, emb_syn),
    }])
else:
    semantic_metrics = pd.DataFrame()

semantic_metrics.to_csv(OUT_DIR / "global_chunk_level_semantic_distance.csv", index=False)
semantic_metrics


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

,comparison,embedding_centroid_cosine,avg_nearest_synthetic_to_original_cosine,avg_nearest_original_to_synthetic_cosine,frechet_embedding_distance,mmd_rbf_embedding_distance
0,original_sentiment_vs_selected_synthetic_senti...,0.951577,0.719549,0.669106,0.334056,0.000387


## Classifier-based separation test

Train the original two-sample classifier to see whether a model can distinguish the empirical and synthetic samples. Strong separation is evidence of distribution shift, not evidence of inaccuracy.


In [8]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

FRENCH_STOP_WORDS = {
    "alors", "au", "aucuns", "aussi", "autre", "avant", "avec", "avoir",
    "bon", "car", "ce", "cela", "ces", "ceux", "chaque", "ci", "comme",
    "comment", "dans", "des", "du", "dedans", "dehors", "depuis", "devrait",
    "doit", "donc", "dos", "début", "elle", "elles", "en", "encore",
    "essai", "est", "et", "eu", "fait", "faites", "fois", "font", "hors",
    "ici", "il", "ils", "je", "juste", "la", "le", "les", "leur", "là",
    "ma", "maintenant", "mais", "mes", "mine", "moins", "mon", "mot",
    "même", "ni", "nommés", "notre", "nous", "nouveaux", "ou", "où",
    "par", "parce", "parole", "pas", "personnes", "peut", "peu", "pièce",
    "plupart", "pour", "pourquoi", "quand", "que", "quel", "quelle",
    "quelles", "quels", "qui", "sa", "sans", "ses", "seulement", "si",
    "sien", "son", "sont", "sous", "soyez", "sujet", "sur", "ta", "tandis",
    "tellement", "tels", "tes", "ton", "tous", "tout", "trop", "très",
    "tu", "voient", "vont", "votre", "vous", "vu", "ça", "étaient", "état",
    "étions", "été", "être"
}

STOP_WORDS_EN_FR = list(ENGLISH_STOP_WORDS.union(FRENCH_STOP_WORDS))

texts = texts_orig + texts_syn
labels = np.array([0] * len(texts_orig) + [1] * len(texts_syn))

c2st_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words=STOP_WORDS_EN_FR,
        ngram_range=(1, 2),
        max_features=30000,
        min_df=2,
    )),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced")),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_validate(
    c2st_pipeline,
    texts,
    labels,
    cv=cv,
    scoring=["accuracy", "f1", "roc_auc"],
)

c2st_results = pd.DataFrame([{
    "comparison": "original_sentiment_vs_selected_synthetic_sentiment_chunks",
    "accuracy_mean": scores["test_accuracy"].mean(),
    "accuracy_std": scores["test_accuracy"].std(),
    "f1_mean": scores["test_f1"].mean(),
    "f1_std": scores["test_f1"].std(),
    "roc_auc_mean": scores["test_roc_auc"].mean(),
    "roc_auc_std": scores["test_roc_auc"].std(),
}])

c2st_results.to_csv(OUT_DIR / "global_chunk_level_c2st.csv", index=False)
c2st_results


,comparison,accuracy_mean,accuracy_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std
0,original_sentiment_vs_selected_synthetic_senti...,0.941148,0.030636,0.936409,0.034218,0.996124,0.004054


## Pooled Global policy contrast

Compare the combined Global policy set with original and supplemented sentiment records. Country-specific results are intentionally excluded.


In [9]:
policy_sentiment_metrics = pd.DataFrame([
    {
        "comparison": "global_policy_vs_original_sentiment",
        "unigram_js_divergence": js_ngram_distance(texts_policy_global, texts_sentiment_original, (1, 1)),
        "bigram_js_divergence": js_ngram_distance(texts_policy_global, texts_sentiment_original, (2, 2)),
        "tfidf_centroid_cosine": tfidf_centroid_cosine(texts_policy_global, texts_sentiment_original),
    },
    {
        "comparison": "global_policy_vs_balanced_sentiment",
        "unigram_js_divergence": js_ngram_distance(texts_policy_global, texts_sentiment_balanced, (1, 1)),
        "bigram_js_divergence": js_ngram_distance(texts_policy_global, texts_sentiment_balanced, (2, 2)),
        "tfidf_centroid_cosine": tfidf_centroid_cosine(texts_policy_global, texts_sentiment_balanced),
    },
])

policy_sentiment_metrics.to_csv(
    OUT_DIR / "global_policy_sentiment_lexical_comparison.csv",
    index=False,
)

policy_sentiment_metrics


,comparison,unigram_js_divergence,bigram_js_divergence,tfidf_centroid_cosine
0,global_policy_vs_original_sentiment,0.235970,0.607595,0.720874
1,global_policy_vs_balanced_sentiment,0.240048,0.644043,0.678039


## Interpretation and export

Summarize measured lexical and semantic differences, note unavailable metrics, and maintain the distinction between observations and generated samples.


In [10]:
def c2st_interpretation(roc_auc):
    if roc_auc < 0.60:
        return "Low distinguishability."
    if roc_auc < 0.75:
        return "Moderate distinguishability."
    return "High distinguishability."


roc_auc = (
    float(c2st_results["roc_auc_mean"].iloc[0])
    if len(c2st_results)
    else float("nan")
)

report = f'''# Global Chunk-Level Synthetic Sentiment Distance Report

## Corpus balance

- Global policy chunks: {len(policy_pool)}
- Original sentiment chunks: {len(orig_sentiment)}
- Synthetic sentiment chunks available: {len(syn_sentiment)}
- Selected synthetic sentiment chunks: {len(syn_balanced)}
- Balanced sentiment chunks: {len(balanced_sentiment)}

## Character balance

- Global policy characters: {policy_pool_chars}
- Original sentiment characters: {orig_sentiment_chars}
- Synthetic characters required: {required_synthetic_chars}
- Selected synthetic characters: {syn_balanced["text_for_eval"].str.len().sum()}
- Balanced sentiment characters: {balanced_sentiment["text_for_eval"].str.len().sum()}

## Original vs selected synthetic sentiment — lexical

{lexical_metrics.to_markdown(index=False)}

## Original vs selected synthetic sentiment — semantic

{semantic_metrics.to_markdown(index=False) if len(semantic_metrics) else "Semantic embeddings were not computed."}

## Original vs selected synthetic sentiment — C2ST

{c2st_results.to_markdown(index=False)}

Interpretation: {c2st_interpretation(roc_auc)}

## Global policy comparison

{policy_sentiment_metrics.to_markdown(index=False)}

## Methodological note

This is a chunk-level validation calibrated to the pooled global policy corpus. The synthetic sentiment corpus is used only as a labelled auxiliary corpus for robustness, sensitivity, and corpus-balance testing.
'''

(OUT_DIR / "global_chunk_level_distance_summary_report.md").write_text(report, encoding="utf-8")
print(report)


# Global Chunk-Level Synthetic Sentiment Distance Report

## Corpus balance

- Global policy chunks: 2052
- Original sentiment chunks: 536
- Synthetic sentiment chunks available: 212
- Selected synthetic sentiment chunks: 212
- Balanced sentiment chunks: 748

## Character balance

- Global policy characters: 3370085
- Original sentiment characters: 900307
- Synthetic characters required: 2469778
- Selected synthetic characters: 446379
- Balanced sentiment characters: 1346686

## Original vs selected synthetic sentiment — lexical

| comparison                                                |   unigram_js_divergence |   bigram_js_divergence |   tfidf_centroid_cosine |
|:----------------------------------------------------------|------------------------:|-----------------------:|------------------------:|
| original_sentiment_vs_selected_synthetic_sentiment_chunks |                0.215477 |               0.562959 |                0.575896 |

## Original vs selected synthetic sentiment — 